# Perfil de um ator

Este notebook mostra, para um único ator, a etapa descrita em `ACTOR_PROFILES.md`: as
falas que entram, o prompt que o modelo recebe, o comando que gera o perfil e a linha que sai
no JSONL. O modelo usado aqui é pequeno o bastante para rodar num notebook comum, e o perfil
que ele escreve serve só para mostrar o formato da saída.

In [1]:
import json
import os
import re
import textwrap
from pathlib import Path

import pandas as pd

from experiments.common.provenance import PROJECT_DIR

os.chdir(PROJECT_DIR)

## Parâmetros

`MODEL` é o mesmo valor que vai em `--model`: id de um repositório do Hugging Face ou caminho de
uma pasta local. `ACTOR` é o nome exato do campo `actor` no arquivo de entrada.

A saída da demonstração fica em `artifacts/cache/actor_profiles/demo.jsonl`, fora do arquivo da
rodada completa (`artifacts/actor_profiles/actor_profiles.jsonl`). O script pula atores que já
têm linha no arquivo de saída, então um perfil de demonstração gravado lá faria a rodada completa
pular este ator.

In [2]:
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
ACTOR = "Raimundo Santos"
CONFIG_PATH = Path("configs/actor_profiles.toml")
OUTPUT_PATH = Path("artifacts/cache/actor_profiles/demo.jsonl")

## As falas do ator

O arquivo de entrada (`actors_multi_hearing.jsonl`, construído como descrito em
`HEARING_ACTORS.md`) tem uma linha por ator recorrente, com as falas agrupadas por audiência e
cada turno marcado como `speaker` ou `chair` (presidência da sessão). As funções de leitura são
as do próprio gerador, para que o notebook veja exatamente o que o script vê.

In [3]:
from experiments.actors.generate_profiles import (
    actor_hearings,
    load_config,
    load_hearing_metadata,
    load_prompts,
)

In [4]:
config = load_config(CONFIG_PATH)
print(config.speeches_path)

artifacts/cache/hearing_actors/actors_multi_hearing.jsonl


In [5]:
with open(config.speeches_path) as f:
    record = next(row for line in f if (row := json.loads(line))["actor"] == ACTOR)

In [6]:
speech_summary = pd.DataFrame(
    [
        {
            "hearing_id": hearing["hearing_id"],
            "turns": len(hearing["turns"]),
            "chair_turns": sum(turn["role"] == "chair" for turn in hearing["turns"]),
            "words": sum(len(turn["text"].split()) for turn in hearing["turns"]),
        }
        for hearing in record["hearings"]
    ]
)
speech_summary

,hearing_id,turns,chair_turns,words
0,166,1,0,395
1,184,1,0,299
2,197,3,2,629


In [7]:
speech_summary[["turns", "chair_turns", "words"]].sum()

turns             5
chair_turns       2
words          1323
dtype: int64

São 5 turnos em 3 audiências, 1.323 palavras no total, e os dois turnos de presidência estão
na audiência 197. O ator foi escolhido por essas duas razões: pouco material deixa a geração
rápida num notebook, e os turnos de presidência mostram a marcação que o template aplica.

## O prompt

O script manda ao modelo duas mensagens: o system prompt fixo
(`prompts/actor_profile/system_profile.md`) e a mensagem do usuário, renderizada pelo template
Jinja2 `user_profile.md.j2` com as falas em ordem cronológica. Cada audiência vem precedida da
data e do assunto, lidos do LDS. As células abaixo refazem essa renderização sem carregar o
modelo. `prompt_version` é o hash dos dois arquivos de prompt e vai gravado em cada perfil.

In [8]:
prompts = load_prompts(config)
print(prompts.version)

a80b3a121fab


In [9]:
hearing_metadata = load_hearing_metadata(config)

In [10]:
user_prompt = prompts.user_profile.render(
    actor_label=record["actor"],
    hearings=actor_hearings(record, hearing_metadata),
)
print(user_prompt)

Escreva o perfil de Raimundo Santos a partir das falas abaixo, transcritas de 3 audiência(s) pública(s), em ordem cronológica.

=== Audiência de 09/08/2023 sobre: Revisão do Marco Legal da Inovação para facilitar sua operacionalização ===

[presidência da sessão]
Declaro aberta a presente reunião de audiência pública da Comissão de Ciência, Tecnologia e Inovação, convocada com o objetivo de discutir os impactos e perspectivas de revisão do novo marco legal da inovação.

Esta audiência pública atende o Requerimento nº 20, de 2023, formulado pela ilustre Deputada Luisa Canziani, Presidente desta Comissão, aprovado em reunião deliberativa da Comissão.

Em razão do número de convidados, as apresentações serão divididas em duas Mesas.

Os procedimentos a serem adotados serão os seguintes: cada palestrante terá 10 minutos para fazer a sua exposição, não sendo permitidos apartes durante a exposição.

Encerradas as apresentações das duas Mesas, será concedida a palavra aos Parlamentares inscri

## Como rodar

A geração é o comando abaixo, rodado dentro de `challenge/`. Sem `--actors`, o script processa
todos os atores do arquivo de entrada; sem `--output`, grava em
`artifacts/actor_profiles/actor_profiles.jsonl`; `--model` pode ser omitido se `name` estiver
preenchido na seção `[model]` de `configs/actor_profiles.toml`.

Como o arquivo de saída é aberto em modo append e os atores que já têm linha nele são pulados, a
primeira célula apaga a saída da demonstração, para que cada execução do notebook gere o perfil
de novo. Na segunda, as linhas de log das requisições HTTP ao Hugging Face são filtradas para
deixar a saída legível.

In [11]:
OUTPUT_PATH.unlink(missing_ok=True)

In [12]:
!uv run python -m experiments.actors.generate_profiles \
    --model {MODEL} \
    --actors "{ACTOR}" \
    --output {OUTPUT_PATH} \
    2>&1 | grep -v "HTTP Request"

INFO 1 actors selected, prompts a80b3a121fab, loading Qwen/Qwen2.5-1.5B-Instruct


WARNING Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


Loading weights: 100%|██████████| 338/338 [00:00<00:00, 782.77it/s]


INFO [1/1] Raimundo Santos: 4,427 in / 2,096 out tokens, 216.6s


done: 1 profiles, 0 failures, 0 skipped (resume)
tokens: 4,427 in / 2,096 out, elapsed 3.6 min


## O resultado

Cada linha do JSONL é um ator. Além do texto em `profile`, a linha registra o modelo, o
`prompt_version`, o volume de material (`n_statements`, `n_hearings`, `hearing_ids`), o tamanho
do prompt e da resposta no tokenizer do modelo e o tempo de geração.

In [13]:
with open(OUTPUT_PATH) as f:
    profile_row = json.loads(f.readline())

In [14]:
pd.Series({key: value for key, value in profile_row.items() if key != "profile"})

actor                          Raimundo Santos
model               Qwen/Qwen2.5-1.5B-Instruct
prompt_version                    a80b3a121fab
n_statements                                 5
n_hearings                                   3
hearing_ids                    [166, 184, 197]
input_tokens                              4427
output_tokens                             2096
generated_at         2026-09-27T02:29:03+00:00
duration_seconds                         216.6
dtype: object

In [15]:
print(profile_row["profile"])

## Posições

**Defende que:**  
- Aumentar a NodeList de pesquisa e desenvolvimento de inovação  
- Implementar sistemas de gestão de inovação  
- Promover a cooperação entre universidades, empresas e governo  
- Aplicar tecnologias como blockchain  
- Garantir a transparência e a integridade dos serviços públicos  
- Proteger a privacidade e a integridade dos dados pessoais  

**Critica que:**  
- O uso excessivo de tecnologia em detrimento da humanidade  
- A falta de regulamentação adequada para blockchain  
- A necessidade de maior investimento em ciência e tecnologia  
- A falta de transparência e controle na administração pública  
- A vulnerabilidade dos dados pessoais diante de vazamentos  
- A falta de políticas específicas para proteção digital  

**Propõe que:**  
- Criar um programa nacional de inovação  
- Implementar um sistema de monitoramento de inovação  
- Estabelecer um conselho de ética para a administração pública  
- Implementar um plano de segurança para blockcha

In [16]:
print(f"palavras no perfil: {len(profile_row['profile'].split())} (limite do prompt: 800)")

palavras no perfil: 1243 (limite do prompt: 800)


## Conferências contra as falas

Duas conferências baratas, sem modelo, apontam onde ler o perfil com atenção. Nenhuma das duas
prova que o perfil está correto: distorções escritas com palavras que existem nas falas passam
por ambas.

A primeira trata de nomes e siglas. O system prompt proíbe acrescentar nomes, siglas, órgãos ou
cargos que as falas não trazem, então a célula lista as palavras do perfil com inicial maiúscula
que não aparecem, em nenhuma capitalização, nas falas do ator nem no nome dele. Os títulos dos
blocos e a primeira palavra de cada item, que é o verbo pedido pelo prompt, ficam fora da
conferência. Início de frase dentro de um item e paráfrase legítima ainda podem cair na lista,
então cada palavra listada precisa ser conferida no texto.

In [17]:
allowed_words = {
    word.lower()
    for hearing in record["hearings"]
    for turn in hearing["turns"]
    for word in re.findall(r"\w+", turn["text"])
} | set(ACTOR.lower().split())

In [18]:
profile_items = [line for line in profile_row["profile"].splitlines() if not line.startswith("## ")]

In [19]:
sorted(
    {
        word
        for line in profile_items
        for word in re.findall(r"\w+", line)[1:]
        if word[0].isupper() and word.lower() not in allowed_words
    }
)

['Jeżeli', 'Kohana', 'NodeList']

A segunda trata do cabeçalho. O `assunto` de cada audiência vem da matéria jornalística, e o
system prompt manda usá-lo só para situar as falas no tempo e no tema, sem atribuir à pessoa o
que aparece apenas nele. A célula lista os assuntos copiados literalmente no perfil; por ser
cópia exata, ela só pega o vazamento mais grosseiro.

In [20]:
[
    (hearing["date"], hearing["assunto"])
    for hearing in actor_hearings(record, hearing_metadata)
    if hearing["assunto"].lower() in profile_row["profile"].lower()
]

[]

Um termo do perfil que pareça suspeito na leitura, mesmo em minúsculas, pode ser procurado
direto nas falas. A célula abaixo lista cada ocorrência de `blockchain` e de `PSD` nas falas do
ator, com o trecho em volta.

In [21]:
[
    (term, hearing["hearing_id"], turn["text"][max(0, match.start() - 120) : match.end() + 60])
    for term in ["blockchain", "PSD"]
    for hearing in record["hearings"]
    for turn in hearing["turns"]
    for match in re.finditer(term, turn["text"], flags=re.IGNORECASE)
]

[('PSD',
  166,
  ' mais para poder ter a oportunidade de cumprimentar V.Exa., que veio a engrandecer sobremaneira não apenas a bancada do PSD e esta Comissão, mas também esta Casa. V.Exa. tem trazido, '),
 ('PSD',
  166,
  'mais avançar sem o auxílio da tecnologia e da inovação.\n\nConfesso a V.Exa., meu querido colega Deputado Caio Vianna, do PSD, e a todos os integrantes — a quem parabenizo pela mais alt')]

Na execução salva deste notebook, o modelo de demonstração não segue o prompt. O perfil tem
1.243 palavras, acima do teto de 800, usa negrito e subtítulos dentro dos blocos, que o prompt
proíbe, e o bloco de forma de argumentar repete os mesmos itens até degenerar em palavras soltas
(`Jeżeli` e `Kohana`, que a primeira conferência acusa junto com `NodeList`). O conteúdo também
viola a fonte única: o perfil fala em blockchain, termo que a última célula não encontra nas falas,
e declara alinhamento com a bancada do PSD, que nas falas só aparece em cumprimentos a um colega,
dedução a partir de elogio que o prompt proíbe no bloco de alinhamentos. A conferência do cabeçalho
não acusa nada. Esses defeitos são do texto que o modelo de demonstração gerou nesta execução; a
qualidade da rodada completa depende do modelo escolhido e precisa ser medida sobre a saída dele.